# Rețele neuronale și deep learning — Seminarul 14: Modele multimodale și vizual-lingvistice; sinteza cursului
Prof. dr. Daniel Traian Pele — Academia de Studii Economice din București (ASE); Academia Română, Institutul de Prognoză Economică (IPE)

[Deschide în Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L14_seminar_ro.ipynb)

Sarcinile seminarului, pe datele reale din `../data/clean` (vezi `../data/SOURCES.md`). Scrieți codul în celulele goale de sub fiecare sarcină; păstrați valoarea fixă a generatorului aleator, ca rezultatele să poată fi comparate.

In [ ]:
# Google Colab: copiem repo-ul cursului, ca datele și rezultatele salvate să fie găsite
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Pregătire

In [ ]:
RES = '../lectures/L14/results/'
import torch, torchvision
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
root = '../data/raw/vision'
os.makedirs(root, exist_ok=True)
ES = torchvision.datasets.EuroSAT(root, download=True)
Z = pd.read_csv(RES + 'eurosat_zeroshot.csv')
CLS = ES.classes
print(len(ES), 'images;', len(Z), 'test images in the lecture split')

In [ ]:
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
def img_emb(idx):
    out = []
    with torch.no_grad():
        for j in range(0, len(idx), 64):
            px = proc(images=[ES[int(i)][0] for i in idx[j:j + 64]], return_tensors='pt')['pixel_values'].to(DEVICE)
            e = clip.get_image_features(pixel_values=px)
            e = e if isinstance(e, torch.Tensor) else e.pooler_output
            out.append(torch.nn.functional.normalize(e.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)
def txt_emb(texts):
    with torch.no_grad():
        t = proc(text=texts, return_tensors='pt', padding=True).to(DEVICE)
        e = clip.get_text_features(**t)
        e = e if isinstance(e, torch.Tensor) else e.pooler_output
        return torch.nn.functional.normalize(e.float(), dim=-1).cpu().numpy()
sample = Z.sample(500, random_state=SEED)
E = img_emb(sample['index'].values)
y = sample['true'].map({c: i for i, c in enumerate(CLS)}).values
NAMES = {'AnnualCrop': 'annual crop land', 'Forest': 'forest', 'HerbaceousVegetation': 'brushland or shrubland', 'Highway': 'highway or road',
         'Industrial': 'industrial buildings or commercial buildings', 'Pasture': 'pasture land', 'PermanentCrop': 'permanent crop land',
         'Residential': 'residential buildings or homes or apartments', 'River': 'river', 'SeaLake': 'lake or sea'}

**Sarcina 1.** Construiți un clasificator zero-shot cu propriul șablon (de exemplu „an aerial image of {}”) și raportați acuratețea pe cele 500 de imagini eșantionate.

In [ ]:
# codul vostru aici

**Sarcina 2.** Faceți media embedding-urilor a trei șabloane pentru fiecare clasă (ansamblu de prompturi). Crește acuratețea față de un singur șablon?

In [ ]:
# codul vostru aici

**Sarcina 3.** Antrenați o regresie logistică pe embedding-urile CLIP ViT-B/32 cu 4 și cu 32 de imagini pe clasă (valoarea generatorului aleator fixă). Reprezentați grafic cele două acurateți lîngă cea zero-shot.

In [ ]:
# codul vostru aici

**Sarcina 4.** Din `eurosat_zeroshot.csv`, listați cele mai frecvente trei confuzii ale celui mai bun model zero-shot.

In [ ]:
# codul vostru aici

**Sarcina 5.** Pentru sarcina regimurilor de piață, calculați un interval bootstrap de 95 % pentru diferența de AUC dintre modelul pe cifre și clasificatorul pe CLIP, pe ferestrele de test.

In [ ]:
# codul vostru aici